# Chapter 11 — Context Management in Production Systems · Homework (self-test)

**Goal:** check that you can pick the structural fix for a context problem and implement the code around it: facts layers, trimming, position-aware input, crash-safe state, server-side context features and session decisions.

- **Theory:** [README.md](README.md) · **Practice:** [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 1.5–2 hours
- **Estimated API cost:** $0. Everything scored runs offline. The optional live exercise uses the free token-counting endpoint.
- **Prerequisites:** you have worked through the practice notebook.

### How to use this notebook

1. Attempt everything **without peeking**. Solutions and explanations are at the very bottom, inside collapsed blocks.
2. **Part A — Quiz:** 11 exam-style scenario questions. Put your letters in the `answers` cell and run the grader.
3. **Part B — Coding exercises:** 6 exercises. Replace each `TODO`, then run the check cell under it. Checks never crash the notebook; they print ✓ or a hint.
4. **Part C — Architecture scenario:** write a short design, then compare it with the rubric.
5. **Scorecard:** the overall score is the average of your quiz % and your exercise %. The pass mark is **72%**, which mirrors the Claude exam's 720/1000.

In [ ]:
import copy
import hashlib
import json
import math
import os
import tempfile
from pathlib import Path
from typing import Any, Callable

import anthropic
import httpx2
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
client = anthropic.Anthropic()  # only the optional live exercise uses it


def est_tokens(obj: Any) -> int:
    """Rough offline estimate (~4 characters per token). Not Claude's tokenizer."""
    s = obj if isinstance(obj, str) else json.dumps(obj, default=str)
    return math.ceil(len(s) / 4)


class ScriptedClaude:
    """A real SDK client whose HTTP layer records each request and answers with a canned reply (offline)."""

    def __init__(self):
        self.requests: list[dict] = []
        transport = httpx2.MockTransport(self._handle)
        self.client = anthropic.Anthropic(api_key="offline", max_retries=0,
                                          http_client=anthropic.DefaultHttpxClient(transport=transport))

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        self.requests.append({"beta": request.headers.get("anthropic-beta"), "body": json.loads(request.content)})
        return httpx2.Response(200, json={
            "id": "msg_x", "type": "message", "role": "assistant", "model": "scripted", "stop_sequence": None,
            "content": [{"type": "text", "text": "ok"}], "stop_reason": "end_turn",
            "usage": {"input_tokens": 0, "output_tokens": 0}})


def make_order(order_id: str, total: float) -> dict:
    """A verbose 43-field order record, like the practice notebook's."""
    raw = {"order_id": order_id, "status": "delivered", "total": total, "currency": "USD", "order_date": "2025-01-15",
           "items": [{"sku": "SKU-BLE", "name": "Blender", "qty": 1, "price": total}], "return_eligible": True,
           "return_window_ends": "2025-02-14",
           "tracking_events": [{"ts": f"2025-01-1{i}T08:00:00Z", "location": f"Hub {i}", "description": "Processed"} for i in range(8)],
           "billing_address": {"line1": "7 Oak Avenue", "city": "Springfield", "zip": "11111"}}
    for i in range(43 - len(raw)):
        raw[f"internal_field_{i:02d}"] = f"value {i} for warehouse, marketing or fraud systems"
    return raw


class Unserializable:  # json.dump fails on this halfway through a write
    pass


# ---- grading helpers ----
RESULTS: dict[int, bool] = {}
QUIZ_RESULT: tuple[int, int] | None = None


def _run_check(n: int, checks: Callable[[], None]) -> None:
    """Run one exercise's asserts and record the result. Never raises."""
    try:
        checks()
    except NotImplementedError:
        RESULTS[n] = False
        print(f"… Exercise {n}: not implemented yet. Replace the TODO and rerun this cell.")
    except AssertionError as err:
        RESULTS[n] = False
        print(f"✗ Exercise {n} not passed yet: {err}")
    except Exception as err:  # a bug in the solution must not crash the notebook
        RESULTS[n] = False
        print(f"✗ Exercise {n} raised {type(err).__name__}: {err}")
    else:
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")


print("Setup done. Model for the optional live exercise:", MODEL)

In [ ]:
# Only the optional live exercise needs a key. Skip this cell if you have no credit.
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"

## Part A — Quiz

Each question has exactly one best answer. Distractors are the anti-patterns from [Common mistakes](README.md#common-mistakes-and-anti-patterns).

**Q1.** A flight-booking assistant stores each conversation in a database. To save tokens, the app rebuilds `messages` for every request from the stored **user** messages only, leaving out the assistant replies and the tool calls with their results. Users notice that the assistant searches again for flights it already found and presented two turns earlier. What is the cause and the fix?

- **A)** The model ignores long histories; add "remember your earlier answers" to the system prompt.
- **B)** The API keeps the assistant's own replies on the server, so the bug must be in the search tool; add retries to it.
- **C)** Claude sees only what the request contains; rebuild `messages` from all stored turns, including assistant replies and each `tool_use` with its `tool_result` (trim large results at ingest).
- **D)** Turn on prompt caching so the earlier turns are remembered between requests.

**Q2.** A support session handles a refund on order ORD-1 ($120) and an exchange on order ORD-2 ($45). At turn 25 the agent issues a $45 refund for ORD-1. The facts block is a flat key/value list (`order_id`, `amount`, `status`) that is overwritten whenever a new tool result arrives. What is the best fix?

- **A)** Restructure the facts into one entry per issue, each with its own order ID, amount and status, and have the agent name the issue it is acting on.
- **B)** Add "double-check amounts before refunding" to the system prompt.
- **C)** Summarize the conversation more often so stale values drop out of context.
- **D)** Move the flat facts list into the top-level system prompt so it carries more authority.

**Q3.** A research team already gave its subagents a JSON output contract with `claim`, `evidence`, `source` and `date` fields. The synthesis input is still about 150K tokens, and a look at the JSON shows that the `evidence` field holds whole scraped pages, HTML included. What should you change?

- **A)** Raise the synthesis agent's `max_tokens` so it has room to read everything.
- **B)** Drop the JSON contract and let the subagents answer in free text, which is usually shorter.
- **C)** Strip the HTML tags from `evidence` and keep the full page text.
- **D)** Tighten the contract: limit `evidence` to a short supporting quote (for example 300 characters) with a source reference, validate it where the result is received, and keep full pages in storage the coordinator can fetch by reference.

**Q4.** A synthesis report presents "AI adoption is 12%" and "AI adoption is 31%" as a contradiction. In fact they come from a 2021 and a 2024 survey. Subagents currently return free-text claims only. Which change fixes this class of error?

- **A)** Instruct the synthesis agent to report the higher value whenever two statistics conflict.
- **B)** Make `date`, `source` and `method` required fields in each subagent's structured output, so synthesis can date and attribute every finding.
- **C)** Have the synthesis agent re-run every web search itself.
- **D)** Put a key-findings summary at the top of the synthesis input.

**Q5.** Halfway through implementing a feature in Claude Code, you need to know which services read the `ORDERS_DB_URL` setting. The last time you asked this kind of question in the main session, Claude grepped the repo, printed hundreds of matching lines, and for the rest of the session its edits drifted from the plan you had agreed. What should you do this time?

- **A)** Ask in the main session, then run `/clear` and explain the feature plan again.
- **B)** Ask the Explore subagent (or a skill with `context: fork`) to find the readers and return a short list of service, file and line; only that list enters the main conversation.
- **C)** Add every config file to CLAUDE.md so Claude never has to search.
- **D)** Run grep yourself in another terminal and paste the full output into the main session.

**Q6.** After a two-hour analysis of a payments module in a named session, the team wants to compare two refactoring approaches (strategy pattern vs event-driven) without one attempt influencing the other. What should they do?

- **A)** Resume the same session ID in two terminals at once without forking.
- **B)** Continue in the same session: try approach 1, then tell Claude to forget it and try approach 2.
- **C)** Start two fresh sessions with no context and redo the analysis in each.
- **D)** Fork the session from the shared analysis once per approach (`claude --resume <id> --fork-session`, or `fork_session=True` in the Agent SDK).

**Q7.** A team adds on-demand compaction (`compaction={"type": "summarize"}`, beta `compact-2026-09-04`). When the call returns `stop_reason: "compaction"`, they append the returned message to the end of their existing history and send the next turn. The API answers 400 `compaction_block_misplaced`. What is the fix?

- **A)** Replace the summarized messages with the returned assistant message so the compaction block comes first, send the block exactly as returned (signature included), and keep the beta header.
- **B)** Repeat the compaction request with a `context_management` edit added.
- **C)** Copy the summary text into a plain user message and drop the beta header.
- **D)** Remove the block's `signature` before sending it back.

**Q8.** A multi-tenant support assistant uses the memory tool with a custom database-backed handler. A penetration test shows that the path `/memories/%2e%2e/tenant-b/notes.md` returns another tenant's notes. What should change?

- **A)** Add a system-prompt rule telling Claude never to read other tenants' memories.
- **B)** Replace the memory tool with server-side context editing.
- **C)** In the handler, decode and resolve every path, reject anything outside that tenant's `/memories` root, and keep storage scoped per tenant.
- **D)** Enable Anthropic's server-side path validation with a beta header.

**Q9.** A team caches a 900K-token knowledge-base prefix on Claude Sonnet 5.5 and expects the full window to stay available for the conversation, because cached tokens "are not processed again". Requests start failing once the conversation reaches about 100K tokens. Why?

- **A)** Prompt caching only works for prompts under 200K tokens.
- **B)** The 5-minute cache expired, so the prefix was sent twice.
- **C)** Cached tokens still occupy the context window. Caching lowers the price and latency of the prefix, not its size, so 900K + 100K reaches the 1M limit.
- **D)** The prefix exceeds the maximum cacheable length.

**Q10.** An internal assistant answers questions over a 3M-token knowledge base that changes daily, plus a 6K-token policy. Each question needs about 1% of the knowledge base. Which context strategy fits best?

- **A)** Cache the stable policy and tool definitions as the prefix, and fetch the relevant knowledge-base slices per question with tools (progressive discovery).
- **B)** Load the whole knowledge base into every request and cache it.
- **C)** Summarize the knowledge base until it fits in the window and load the summary into every request.
- **D)** Load as much of the knowledge base as fits into the system prompt and put a key-findings summary at the top.

**Q11.** A developer has used one Claude Code session all morning: first a refund bug, then an unrelated CSS question, then back to the refund bug. Claude has now failed to fix the bug after three corrections, and its suggestions mix in details from the CSS discussion. What is the best session-hygiene move?

- **A)** Keep correcting in the same session, adding "ignore the CSS discussion" to each message.
- **B)** Run `/clear` and restart the refund task with a sharper prompt that includes what the failed attempts taught you; next time, `/clear` between unrelated tasks.
- **C)** Run `/compact` so the failed attempts and the CSS discussion are summarized, then continue.
- **D)** Switch to a model with a larger context window so the earlier attempts no longer crowd out the task.

In [ ]:
# Fill in one letter per question: "A", "B", "C" or "D".
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
    "q11": None,
}

In [ ]:
ANSWER_KEY = {
    "q1": "227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e",
    "q2": "3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45",
    "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
    "q4": "61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa",
    "q5": "fbb43cbbf51767ad8ca178fa6cc4731661f1f1771789b9b7df93fba7ad386e2c",
    "q6": "70425e0bfb4a573a030df7909d2bae441e3fdb1e7a565e097f3118231f4f27a9",
    "q7": "8722338db4eb12eaae5f6f26c68875af2b115581d770f01f98b9e0e1e6010ac1",
    "q8": "89443923f22dc808ba069cfb289117719cbe7d8d36703c74314c6b70fba0371b",
    "q9": "e19608e84d37f3c94b5f88738c75a8c96772a5e61784e3e24b5512fa65ff63c7",
    "q10": "aeb5147fa5276b8759290ab8af9ed3aaa01d2c844cecc89ea5547b013e209ad3",
    "q11": "42310be80adadfed411c1783b5c42432624ffc39d2ea2f7d837996726d6b0774",
}  # sha256 of "qid:LETTER", so the answers are not readable here


def grade_quiz(answers: dict) -> None:
    global QUIZ_RESULT
    correct = 0
    for qid, expected_hash in ANSWER_KEY.items():
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) and given.strip() else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == expected_hash
        correct += ok
        print(f"{qid:>3}: {'✓' if ok else '✗'}" + ("  (unanswered)" if letter is None else ""))
    QUIZ_RESULT = (correct, len(ANSWER_KEY))
    print(f"\nQuiz score: {correct}/{len(ANSWER_KEY)} = {correct / len(ANSWER_KEY):.0%}")


grade_quiz(answers)

## Part B — Coding exercises

All six run offline. The setup cell defines `make_order` (a 43-field order record), `est_tokens` (a rough estimate, not Claude's tokenizer), `Unserializable` and `ScriptedClaude` (a real SDK client with a fake HTTP layer, used by the Exercise 5 checker).

### Exercise 1 — Merge case facts without losing the verified value

Implement `upsert_fact(issue, key, new)` for a per-issue facts layer.

- `issue` is `{"facts": {key: fact}, "conflicts": [str, ...]}`. A fact is `{"value": str, "source": str, "turn": int, "verified": bool}`. Mutate `issue` in place and return `None`.
- **No existing fact** under `key`: store `new`.
- **Same value**: no conflict. Replace the stored fact only if that upgrades it from unverified to verified.
- **Different value**: append one string to `issue["conflicts"]` that contains the key, the old value and the new value. Then keep the **old** fact if it is verified and the new one is not (a customer claim never overwrites system data); otherwise store the new fact.

Each issue is its own dict, so values never leak between issues.

[Theory: 11.1](README.md#111-extract-facts-into-a-persistent-case-facts-block)

In [ ]:
def upsert_fact(issue: dict, key: str, new: dict) -> None:
    # TODO: store / keep / flag a conflict according to the rules above
    raise NotImplementedError

In [ ]:
def check_ex1():
    def checks():
        def fact(value, source="lookup_order", turn=3, verified=True):
            return {"value": value, "source": source, "turn": turn, "verified": verified}

        i1, i2 = {"facts": {}, "conflicts": []}, {"facts": {}, "conflicts": []}
        assert upsert_fact(i1, "amount", fact("74.95")) is None, "mutate the issue in place and return None"
        assert i1["facts"].get("amount", {}).get("value") == "74.95", "a new key must be stored"
        upsert_fact(i1, "amount", fact("74.95", turn=7))
        assert i1["conflicts"] == [], "the same value again is not a conflict"

        upsert_fact(i1, "amount", fact("99.99", source="customer", turn=22, verified=False))
        assert len(i1["conflicts"]) == 1, "a different value must add exactly one conflict line"
        c = i1["conflicts"][0]
        assert "amount" in c and "74.95" in c and "99.99" in c, f"the conflict line needs the key, old and new value, got {c!r}"
        assert i1["facts"]["amount"]["value"] == "74.95", "an unverified claim must not overwrite a verified value"

        upsert_fact(i1, "promised", fact("25% discount", source="customer", turn=9, verified=False))
        upsert_fact(i1, "promised", fact("10% discount", source="crm", turn=12, verified=True))
        assert len(i1["conflicts"]) == 2, "unverified 25% vs verified 10% is still a conflict to record"
        assert i1["facts"]["promised"]["value"] == "10% discount", "a verified value replaces an unverified claim"

        upsert_fact(i1, "request", fact("full refund", source="customer", verified=False))
        upsert_fact(i1, "request", fact("full refund", source="crm", verified=True))
        assert i1["facts"]["request"]["verified"] is True and len(i1["conflicts"]) == 2, \
            "same value, now verified: upgrade the fact without a conflict"

        upsert_fact(i1, "address", fact("7 Oak Ave"))
        upsert_fact(i1, "address", fact("42 Elm St", source="crm", turn=30))
        assert i1["facts"]["address"]["value"] == "42 Elm St", "a newer verified value replaces an older verified one (after flagging)"

        upsert_fact(i2, "amount", fact("34.50"))
        assert i2["conflicts"] == [] and i1["facts"]["amount"]["value"] == "74.95", "issues must stay independent"
    _run_check(1, checks)


check_ex1()

### Exercise 2 — Trim a tool payload at ingest

Implement `trim_payload(raw, fields, store, ref_prefix="order_raw")`.

- Return a **new** dict with only the keys from `fields` that exist in `raw`, in the order of `fields` (skip missing ones; do not add `None`).
- Add `"full_record_ref": f"{ref_prefix}/{raw['order_id']}"`.
- Save the untouched raw record in `store[raw["order_id"]]` so nothing is lost.
- Do not mutate `raw`.

[Theory: 11.2](README.md#112-trim-verbose-tool-results)

In [ ]:
RETURN_FIELDS = ("order_id", "status", "total", "currency", "order_date", "items", "return_eligible", "return_window_ends")


def trim_payload(raw: dict, fields: tuple[str, ...], store: dict, ref_prefix: str = "order_raw") -> dict:
    # TODO: whitelist projection + reference, raw record kept in `store`
    raise NotImplementedError

In [ ]:
def check_ex2():
    def checks():
        raw = make_order("ORD-30417", 74.95)
        snapshot = json.dumps(raw, sort_keys=True)
        store = {}
        slim = trim_payload(raw, RETURN_FIELDS, store)
        assert isinstance(slim, dict), "return a dict"
        assert json.dumps(raw, sort_keys=True) == snapshot, "do not mutate the raw record"
        assert slim is not raw, "return a new dict"
        expected_keys = list(RETURN_FIELDS) + ["full_record_ref"]
        assert list(slim) == expected_keys, f"keys must be the whitelist (in order) + full_record_ref, got {list(slim)}"
        assert slim["total"] == 74.95 and slim["items"] == raw["items"], "copy the values exactly"
        assert slim["full_record_ref"] == "order_raw/ORD-30417", f"wrong reference: {slim.get('full_record_ref')!r}"
        assert store.get("ORD-30417") == raw, "keep the full raw record in the store"
        assert est_tokens(slim) < 0.25 * est_tokens(raw), "the trimmed result should be a small fraction of the raw payload"

        sparse = {"order_id": "ORD-1", "status": "shipped", "noise": "x" * 500}
        s2 = trim_payload(sparse, RETURN_FIELDS, store, ref_prefix="orders")
        assert s2 == {"order_id": "ORD-1", "status": "shipped", "full_record_ref": "orders/ORD-1"}, \
            f"skip fields the payload does not have (no None values) and honor ref_prefix, got {s2}"
    _run_check(2, checks)


check_ex2()

### Exercise 3 — Key findings first: build a synthesis input

Implement `build_synthesis_input(findings, question)`. Each finding is `{"source", "date", "method", "key_points": [str, ...], "detail": str}`.

1. If any finding has an empty or missing `source` or `date`, raise `ValueError` (metadata is required at the boundary).
2. Start with a `<key_findings>` … `</key_findings>` block that lists **every** key point as `- [i] <point> (<source>, <date>)`, numbering findings from 1.
3. Then the details: each finding inside `<document index="i">` … `</document>`, containing `<source>…</source>`, `<date>…</date>` and its `detail`.
4. End with `<task>` … `</task>` containing the question. Nothing may follow `</task>`.

[Theory: 11.3](README.md#113-position-aware-input-lost-in-the-middle)

In [ ]:
def build_synthesis_input(findings: list[dict], question: str) -> str:
    # TODO: validate metadata; key findings first, <document> sections, task last
    raise NotImplementedError

In [ ]:
def check_ex3():
    def checks():
        filler = "supporting detail " * 300
        findings = [
            {"source": "web-search", "date": "2024-03-01", "method": "survey", "key_points": ["Adoption grew to 31%"], "detail": filler + "A1"},
            {"source": "doc-analysis", "date": "2024-06-30", "method": "report", "key_points": ["Bookings fell 20%", "Rates flat"], "detail": filler + "B2"},
            {"source": "legal", "date": "2024-09-15", "method": "review", "key_points": ["Three lawsuits pending"], "detail": filler + "C3"},
        ]
        q = "How is AI changing session work?"
        out = build_synthesis_input(findings, q)
        assert isinstance(out, str), "return the prompt as a string"
        s = out.strip()
        assert s.startswith("<key_findings>"), "the input must START with the <key_findings> block"
        kf_end = s.find("</key_findings>")
        assert kf_end > 0, "close the </key_findings> block"
        key_block = s[:kf_end]
        for i, f in enumerate(findings, 1):
            for pt in f["key_points"]:
                assert f"[{i}] {pt}" in key_block, f"key findings must list '[{i}] {pt}'"
            assert f["source"] in key_block and f["date"] in key_block, f"key findings need source and date for finding {i}"
        for i, f in enumerate(findings, 1):
            start = s.find(f'<document index="{i}">')
            assert start > kf_end, f'<document index="{i}"> must come after the key findings'
            end = s.find("</document>", start)
            body = s[start:end]
            assert f"<source>{f['source']}</source>" in body and f"<date>{f['date']}</date>" in body, f"document {i} needs <source> and <date>"
            assert f["detail"] in body, f"document {i} must contain its detail"
        last_doc = s.rfind("</document>")
        task_at = s.rfind("<task>")
        assert task_at > last_doc, "the <task> block must come after all documents"
        assert q in s[task_at:], "the question goes inside <task>"
        assert s.endswith("</task>"), "nothing may follow </task>"
        for missing in ("source", "date"):
            bad = [dict(findings[0]), dict(findings[1], **{missing: ""})]
            try:
                build_synthesis_input(bad, q)
            except ValueError:
                pass
            else:
                raise AssertionError(f"a finding with an empty {missing!r} must raise ValueError")
    _run_check(3, checks)


check_ex3()

### Exercise 4 — Atomic checkpoint and a resume plan

Two functions for crash recovery.

**(a)** `write_json_atomic(path, data)`: write `data` as JSON so that a failure halfway through (for example `json.dump` raising `TypeError` on an unserializable value) leaves the **previous** file intact and leaves **no temporary file** behind. Create parent directories if needed. Hint: `tempfile.mkstemp(dir=path.parent)`, then `os.replace`.

**(b)** `resume_plan(manifest, load_state)`: `manifest["agents"]` maps agent name → `{"status": ..., "state_file": ...}`; `load_state(path) -> dict` reads one agent's state (`key_findings`, `gaps`, ...). Return `(todo, injected)`:
- `todo`: names whose status is not `"completed"`, in manifest order (`in_progress` and `failed` agents are re-run).
- `injected`: one string containing, for every **completed** agent, its name, `json.dumps(state["key_findings"])` and its `gaps`. Do not load state for agents that are not completed.

[Theory: 11.6](README.md#116-structured-state-persistence-and-crash-recovery)

In [ ]:
def write_json_atomic(path: Path, data: dict) -> None:
    # TODO: temp file in the same directory, then os.replace; clean up on failure
    raise NotImplementedError


def resume_plan(manifest: dict, load_state: Callable[[str], dict]) -> tuple[list[str], str]:
    # TODO: (todo names in order, injected findings of completed agents)
    raise NotImplementedError

In [ ]:
def check_ex4():
    def checks():
        d = Path(tempfile.mkdtemp(prefix="hw11-"))
        p = d / "nested" / "manifest.json"
        assert write_json_atomic(p, {"status": "completed", "n": 1}) is None, "return None"
        assert json.loads(p.read_text()) == {"status": "completed", "n": 1}, "the file must contain the JSON data"
        write_json_atomic(p, {"status": "completed", "n": 2})
        assert json.loads(p.read_text())["n"] == 2, "a second write must replace the file"
        try:
            write_json_atomic(p, {"status": "in_progress", "bad": Unserializable()})
        except TypeError:
            pass
        else:
            raise AssertionError("let the TypeError from json.dump propagate (do not swallow it)")
        try:
            content = json.loads(p.read_text())
        except json.JSONDecodeError:
            raise AssertionError("a failed write corrupted the previous file: write to a temp file, then os.replace")
        assert content["n"] == 2, "a failed write must leave the previous file intact"
        leftovers = [f.name for f in p.parent.iterdir() if f.name != "manifest.json"]
        assert not leftovers, f"a failed write must not leave temp files behind: {leftovers}"

        states = {"s/web.json": {"key_findings": [{"claim": "AI use grew to 31%"}], "gaps": ["licensing"]},
                  "s/docs.json": {"key_findings": [{"claim": "Revenue fell 8%"}], "gaps": []}}
        loaded = []

        def load_state(path):
            loaded.append(path)
            if path not in states:
                raise FileNotFoundError(path)
            return states[path]

        manifest = {"agents": {
            "web-search": {"status": "completed", "state_file": "s/web.json"},
            "doc-analysis": {"status": "in_progress", "state_file": "s/missing.json"},
            "fact-check": {"status": "failed", "state_file": "s/missing2.json"},
            "docs-2": {"status": "completed", "state_file": "s/docs.json"},
            "synthesis": {"status": "not_started", "state_file": "s/syn.json"},
        }}
        todo, injected = resume_plan(manifest, load_state)
        assert todo == ["doc-analysis", "fact-check", "synthesis"], f"todo must be the non-completed agents in order, got {todo}"
        assert set(loaded) == {"s/web.json", "s/docs.json"}, "only load state for completed agents"
        assert isinstance(injected, str), "injected must be a string"
        for name in ("web-search", "docs-2"):
            assert name in injected, f"injected must name completed agent {name}"
        assert json.dumps(states["s/web.json"]["key_findings"]) in injected, "inject json.dumps(key_findings) of each completed agent"
        assert "licensing" in injected, "inject the completed agents' gaps"
        assert "doc-analysis" not in injected, "do not inject anything for agents that still need to run"
    _run_check(4, checks)


check_ex4()

### Exercise 5 — Pick the right server-side context feature

Implement `plan_request(situation)`. It returns the **extra keyword arguments** for `client.beta.messages.create(model=..., max_tokens=..., messages=..., **extra)`: always a `betas` list, plus either `context_management` or `compaction` (never both).

| `situation` | What the request must do |
|---|---|
| `"long_tool_loop"` | Clear old tool results once the input passes **80,000** tokens, keep the last **5** tool uses, clear at least **20,000** tokens each time, and never clear `get_customer`. |
| `"long_tool_loop_with_thinking"` | The same, and also keep thinking blocks from only the last **2** assistant turns. |
| `"compact_now"` | Summarize the conversation now, on demand, with instructions that mention preserving **order IDs** and **amounts**. |
| `"auto_compact"` | Compact automatically inside a normal reply when the input reaches **120,000** tokens. |

Use the exact strategy type names and beta headers from README 11.9–11.10 and practice 11.9–11.10 (the practice cells show the full `clear_thinking_20251015` shape and that it must be listed first). Raise `ValueError` for an unknown situation. The checker also sends your kwargs through a real SDK client (offline) to make sure the SDK accepts them.

[Theory: 11.9](README.md#119-server-side-compaction-beyond-the-guide) · [11.10](README.md#1110-context-editing-clearing-tool-results-and-thinking-beyond-the-guide)

In [ ]:
def plan_request(situation: str) -> dict:
    # TODO: return {"betas": [...], "context_management": {...}} or {"betas": [...], "compaction": {...}}
    raise NotImplementedError

In [ ]:
def check_ex5():
    def checks():
        def edits_of(kw):
            return kw.get("context_management", {}).get("edits", [])

        for s in ("long_tool_loop", "long_tool_loop_with_thinking", "compact_now", "auto_compact"):
            kw = plan_request(s)
            assert isinstance(kw, dict) and isinstance(kw.get("betas"), list), f"{s}: return a dict with a 'betas' list"
            assert not ("compaction" in kw and "context_management" in kw), f"{s}: compaction and context_management cannot be combined"
            fake = ScriptedClaude()
            fake.client.beta.messages.create(model=MODEL, max_tokens=1024, messages=[{"role": "user", "content": "hi"}], **kw)
            assert fake.requests, f"{s}: the SDK did not send the request"

        kw = plan_request("long_tool_loop")
        assert kw["betas"] == ["context-management-2025-06-27"], f"long_tool_loop: context editing beta header expected, got {kw['betas']}"
        e = edits_of(kw)
        assert len(e) == 1 and e[0].get("type") == "clear_tool_uses_20250919", "long_tool_loop: one clear_tool_uses_20250919 edit"
        assert e[0].get("trigger") == {"type": "input_tokens", "value": 80_000}, "trigger: input_tokens 80,000"
        assert e[0].get("keep") == {"type": "tool_uses", "value": 5}, "keep: the last 5 tool uses"
        assert e[0].get("clear_at_least") == {"type": "input_tokens", "value": 20_000}, "clear_at_least: 20,000 input tokens"
        assert e[0].get("exclude_tools") == ["get_customer"], "exclude_tools: never clear get_customer"

        e = edits_of(plan_request("long_tool_loop_with_thinking"))
        assert len(e) == 2, "with thinking: two edits"
        assert e[0].get("type") == "clear_thinking_20251015", "clear_thinking_20251015 must be listed FIRST"
        assert e[0].get("keep") == {"type": "thinking_turns", "value": 2}, "keep thinking from the last 2 turns"
        assert e[1].get("type") == "clear_tool_uses_20250919", "then the tool-result clearing edit"

        kw = plan_request("compact_now")
        assert kw["betas"] == ["compact-2026-09-04"], f"on-demand compaction beta header expected, got {kw['betas']}"
        assert kw.get("compaction", {}).get("type") == "summarize", "on-demand compaction uses compaction={'type': 'summarize', ...}"
        instr = (kw["compaction"].get("instructions") or "").lower()
        assert "order id" in instr and "amount" in instr, "instructions should ask to preserve order IDs and amounts"
        assert len(instr) <= 16_384, "instructions are limited to 16,384 characters"

        kw = plan_request("auto_compact")
        assert kw["betas"] == ["compact-2026-01-12"], f"threshold compaction beta header expected, got {kw['betas']}"
        e = edits_of(kw)
        assert len(e) == 1 and e[0].get("type") == "compact_20260112", "auto_compact: one compact_20260112 edit"
        assert e[0].get("trigger") == {"type": "input_tokens", "value": 120_000}, "trigger at 120,000 input tokens"

        try:
            plan_request("make_it_smaller")
        except ValueError:
            pass
        else:
            raise AssertionError("raise ValueError for an unknown situation")
    _run_check(5, checks)


check_ex5()

### Exercise 6 — Measure the window, then choose resume, fork or fresh

Two small decision helpers.

**(a)** `window_used(usage)`: everything that occupied the context window on one request, from an SDK `Usage` object: `input_tokens + cache_read_input_tokens + cache_creation_input_tokens + output_tokens`. The cache fields are `Optional` and can be `None`.

**(b)** `choose_session(changed_files, compare_approaches=False, few=3)` returns one of `"fork"`, `"resume"`, `"resume_with_changes"`, `"fresh_with_summary"`:
- comparing two approaches from the same baseline → `"fork"` (takes priority);
- `changed_files is None` (unknown changes) or more than `few` changed files → `"fresh_with_summary"`;
- 1 to `few` known changed files → `"resume_with_changes"`;
- no changes (`[]`) → `"resume"`.

[Theory: 11.8](README.md#118-context-windows-token-budgets-and-counting-beyond-the-guide) · [11.7](README.md#117-sessions-resume-fork-or-start-fresh-task-17)

In [ ]:
def window_used(usage) -> int:
    # TODO: sum all input fields (None-safe) and the output
    raise NotImplementedError


def choose_session(changed_files: list[str] | None, compare_approaches: bool = False, few: int = 3) -> str:
    # TODO: fork / fresh_with_summary / resume_with_changes / resume
    raise NotImplementedError

In [ ]:
def check_ex6():
    def checks():
        U = anthropic.types.Usage
        assert window_used(U(input_tokens=1_000, output_tokens=200)) == 1_200, "no cache fields (None) must count as 0"
        warm = U(input_tokens=900, cache_read_input_tokens=17_000, cache_creation_input_tokens=100, output_tokens=600)
        got = window_used(warm)
        assert got == 18_600, f"cached tokens still occupy the window: expected 18,600, got {got}"
        assert window_used(U(input_tokens=5, cache_read_input_tokens=0, cache_creation_input_tokens=None, output_tokens=0)) == 5, \
            "handle a mix of 0 and None"

        cases = [
            (([], False), "resume", "nothing changed"),
            ((["a.ts"], False), "resume_with_changes", "one known file changed"),
            ((["a.ts", "b.ts", "c.ts"], False), "resume_with_changes", "exactly `few` files is still 'few'"),
            ((["a", "b", "c", "d"], False), "fresh_with_summary", "more than `few` files changed"),
            ((None, False), "fresh_with_summary", "unknown changes"),
            (([], True), "fork", "comparing two approaches"),
            ((None, True), "fork", "fork takes priority"),
        ]
        for (files, compare), expected, why in cases:
            got = choose_session(files, compare)
            assert got == expected, f"{why}: expected {expected!r}, got {got!r}"
        assert choose_session(["a", "b"], few=1) == "fresh_with_summary", "respect the `few` parameter"
    _run_check(6, checks)


check_ex6()

### Optional live exercise (live — uses the free `count_tokens` endpoint, not scored)

Compare the real token count of a raw `lookup_order` result with your trimmed version from Exercise 2.

In [ ]:
raw = make_order("ORD-30417", 74.95)
variants = {"raw": raw}
try:
    variants["trimmed"] = trim_payload(raw, RETURN_FIELDS, {})
except NotImplementedError:
    print("Finish Exercise 2 first; counting the raw payload only.")

for name, payload in variants.items():
    messages = [
        {"role": "user", "content": "Can I return ORD-30417?"},
        {"role": "assistant", "content": [{"type": "tool_use", "id": "toolu_1", "name": "lookup_order", "input": {"order_id": "ORD-30417"}}]},
        {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "toolu_1", "content": json.dumps(payload)}]},
    ]
    tools = [{"name": "lookup_order", "description": "Look up an order by ID.",
              "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}]
    n = client.messages.count_tokens(model=MODEL, tools=tools, messages=messages).input_tokens
    print(f"{name:<8} {n:>6} input tokens  (offline estimate {est_tokens(messages):>5})")

## Part C — Architecture scenario

**Scenario.** A telecom runs a customer-support assistant on Claude Sonnet 5.5 with thinking enabled. Chats average 45 turns and often cover up to three issues at once (a billing dispute, a plan change, a device return). The CRM tool returns 60-field customer and order records. A 25K-token policy document is updated monthly. Customers often come back the next day, after the CRM has changed. Separately, a nightly multi-agent pipeline analyzes about 200 documents for churn risk; it takes about 50 minutes and sometimes crashes halfway.

Design the **context architecture** for both. Cover:

1. The layers of each support request and where each one sits (what is cached, what changes per turn).
2. How exact values (amounts, dates, promises) survive a 45-turn, multi-issue chat.
3. How you keep tool output from dominating the context, now and late in a long tool loop.
4. What happens when a chat approaches your token budget.
5. Next-day returns: resume, or start fresh?
6. How the nightly pipeline survives a crash, and how its agents hand results to synthesis.
7. What you measure to know the design is working.

Write your answer in the cell below (about 250–350 words).

*Your answer:*

<details><summary>Rubric: what a strong answer contains (open after you have written yours)</summary>

- **Layered request:** static policy + tool definitions first, cached (`cache_control` breakpoint; `tools` declared up front and never edited mid-chat); summary of older turns; recent turns verbatim; trimmed tool results; the case-facts block; the customer's message last. Changing state never goes into the top-level `system` prompt.
- **Case-facts block, per issue:** order/account IDs, amounts, dates, plan names, customer-stated expectations and agent promises, each with source and turn; values copied from tool results in code, customer claims marked unverified; conflicts flagged, never silently overwritten. Sent every turn in the newest user turn and never summarized. A turn-scoped system message suits only verified, system-sourced facts, because system text carries operator authority; customer claims stay in the user turn.
- **Tool output:** trim the 60-field records at ingest (in the tool, the loop or a `PostToolUse` hook) to the fields each task needs, with a reference to the raw record stored outside the window. Late in long tool loops, server-side **context editing** (`clear_tool_uses_20250919` with `clear_at_least` and `exclude_tools`), never client-side edits to old turns (they invalidate preserved thinking).
- **Budget:** a per-layer token budget, `count_tokens` before sending, all `usage` fields after (cached tokens still count). Near the budget: **server-side compaction** with instructions to preserve values, signed block kept first, the facts block still outside the summary. Recognizes that a 1M window does not remove the need (cost, context rot).
- **Next-day returns:** CRM data changed, so either a fresh session seeded with a structured summary plus the facts block re-verified against the CRM, or resume with an explicit list of what changed. Not a blind resume.
- **Nightly pipeline:** per-agent structured state files + a manifest, written atomically after every status change; on restart, skip completed agents, re-run in-progress ones idempotently, inject completed findings into remaining prompts. Upstream agents return structured findings with source, date and method (no raw pages), and synthesis input puts key findings first, details under headers, the task last.
- **Metrics:** tokens per turn and per chat, cache hit rate (`cache_read_input_tokens` share), compaction frequency, fact-accuracy checks (does the agent quote the right amount/promise late in chats), wrong-issue actions, pipeline resume success and time saved.
- Bonus: memory tool or CRM notes for cross-session preferences (path-validated, per-customer); retrieval over past chats for "what did we agree last month".

</details>

## Scorecard

In [ ]:
quiz_correct, quiz_total = QUIZ_RESULT if QUIZ_RESULT else (0, 11)
ex_total = 6
ex_passed = sum(RESULTS.get(n, False) for n in range(1, ex_total + 1))
quiz_pct, ex_pct = quiz_correct / quiz_total, ex_passed / ex_total
overall = (quiz_pct + ex_pct) / 2

print(f"Quiz       : {quiz_correct}/{quiz_total} ({quiz_pct:.0%})" + ("" if QUIZ_RESULT else "  <- run the quiz grader"))
print(f"Exercises  : {ex_passed}/{ex_total} passed ({ex_pct:.0%})   " +
      " ".join(f"ex{n}:{'✓' if RESULTS.get(n) else '✗'}" for n in range(1, ex_total + 1)))
print(f"Overall    : {overall:.0%}  ->  {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%)")

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers and explanations</summary>

**Q1: C.** The Messages API keeps no conversation state: Claude knows only what the current request contains, so a history without the assistant turns and tool results hides the search it already did. A asks the model to remember something it was never sent. B describes server-side memory the Messages API does not have. D changes what the tokens cost; a cache only holds what you send.

**Q2: A.** Parameters leak between issues because one flat layer holds both. Per-issue entries keep each order's values separate (task 5.1, S2). B is a vague instruction against a structural bug. C makes exact values less available, not more. D changes authority, not structure, and re-rendering `system` every turn also breaks the prompt cache.

**Q3: D.** A contract only helps if its fields are tight: here the bloat comes from upstream, so fix it at the source (task 5.1, S6). A raises the output limit, not the input size. B loses the source and date fields synthesis needs. C removes markup but still forwards whole pages.

**Q4: B.** Synthesis can only interpret a finding if the metadata travels with it (task 5.1, S5). A hides a real temporal trend behind an arbitrary rule. C duplicates work and floods the synthesis context. D fixes lost-in-the-middle, not missing dates.

**Q5: B.** Isolating verbose discovery prevents the damage instead of cleaning it up (tasks 3.4 and 5.4). A throws away the feature plan along with the noise. C loads config noise into every future session. D puts the same flood of lines into the main context.

**Q6: D.** Forking gives each approach the same baseline and an independent branch (task 1.7, K2). A has both terminals write to one session, so the two approaches mix. B leaves approach 1 in context. C throws away two hours of valid analysis.

**Q7: A.** The block replaces the messages it summarized and must come first. B is rejected: `compaction` and `context_management` cannot be combined on one request. C abandons on-demand compaction: the signed block is discarded and the request becomes a plain client-side summary, which does not fix the team's compaction flow. D makes the block invalid, because it must be sent back exactly as returned.

**Q8: C.** The memory tool is a client tool: your application executes every command, so path validation and tenant scoping are your job. A is a probabilistic control for a security boundary. B solves a different problem (clearing old tool results). D does not exist.

**Q9: C.** Caching changes what you pay, not whether tokens count toward the window. A and D are invented limits. B is wrong: an expired cache means the prefix is written to the cache again on the next request (1.25× base input for the 5-minute cache); it is not sent twice and the window size is unchanged.

**Q10: A.** A large, changing corpus where each request needs a small slice is the textbook case for a cached stable core plus just-in-time retrieval (CCAR-P task 3.8). B does not fit the window and would be rewritten daily. C loses the specifics questions need. D still fills the window with mostly irrelevant content and changes the cached prefix whenever the corpus changes.

**Q11: B.** This is the "kitchen sink" session plus repeated corrections from the Claude Code best practices: the context is polluted with an unrelated task and failed approaches, and a clean session with a better prompt beats a long one (CCDV-F 2.5, session hygiene). A keeps the pollution in context. C keeps a summary of the failed approaches and the unrelated topic. D keeps all the noise in view.

</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
def upsert_fact(issue: dict, key: str, new: dict) -> None:
    old = issue["facts"].get(key)
    if old is None:
        issue["facts"][key] = new
        return
    if old["value"] == new["value"]:
        if new["verified"] and not old["verified"]:
            issue["facts"][key] = new          # same value, now confirmed by a system
        return
    issue["conflicts"].append(f"{key}: {old['value']} ({old['source']}, t{old['turn']}) vs "
                              f"{new['value']} ({new['source']}, t{new['turn']})")
    if old["verified"] and not new["verified"]:
        return                                 # a claim never overwrites verified data
    issue["facts"][key] = new
```

</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
RETURN_FIELDS = ("order_id", "status", "total", "currency", "order_date", "items", "return_eligible", "return_window_ends")


def trim_payload(raw: dict, fields: tuple[str, ...], store: dict, ref_prefix: str = "order_raw") -> dict:
    slim = {k: raw[k] for k in fields if k in raw}
    store[raw["order_id"]] = raw
    slim["full_record_ref"] = f"{ref_prefix}/{raw['order_id']}"
    return slim
```

</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
def build_synthesis_input(findings: list[dict], question: str) -> str:
    for f in findings:
        if not f.get("source") or not f.get("date"):
            raise ValueError(f"finding without source/date: {f.get('key_points')}")
    summary = "\n".join(f"- [{i}] {pt} ({f['source']}, {f['date']})"
                        for i, f in enumerate(findings, 1) for pt in f["key_points"])
    sections = "\n".join(
        f'<document index="{i}">\n<source>{f["source"]}</source>\n<date>{f["date"]}</date>\n'
        f'<method>{f.get("method", "")}</method>\n<document_content>\n{f["detail"]}\n</document_content>\n</document>'
        for i, f in enumerate(findings, 1))
    return (f"<key_findings>\n{summary}\n</key_findings>\n\n<detailed_results>\n{sections}\n</detailed_results>\n\n"
            f"<task>\n{question}\nCite findings by [index].\n</task>")
```

</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
def write_json_atomic(path: Path, data: dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(data, f, indent=2)
        os.replace(tmp, path)
    except BaseException:
        os.unlink(tmp)
        raise


def resume_plan(manifest: dict, load_state: Callable[[str], dict]) -> tuple[list[str], str]:
    todo, parts = [], []
    for name, entry in manifest["agents"].items():
        if entry["status"] != "completed":
            todo.append(name)
            continue
        state = load_state(entry["state_file"])
        parts.append(f"<prior_results agent='{name}'>\n{json.dumps(state['key_findings'])}\n"
                     f"gaps={state.get('gaps', [])}\n</prior_results>")
    return todo, "\n".join(parts)
```

</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def plan_request(situation: str) -> dict:
    clear_tools = {
        "type": "clear_tool_uses_20250919",
        "trigger": {"type": "input_tokens", "value": 80_000},
        "keep": {"type": "tool_uses", "value": 5},
        "clear_at_least": {"type": "input_tokens", "value": 20_000},
        "exclude_tools": ["get_customer"],
    }
    if situation == "long_tool_loop":
        return {"betas": ["context-management-2025-06-27"], "context_management": {"edits": [clear_tools]}}
    if situation == "long_tool_loop_with_thinking":
        thinking = {"type": "clear_thinking_20251015", "keep": {"type": "thinking_turns", "value": 2}}
        return {"betas": ["context-management-2025-06-27"], "context_management": {"edits": [thinking, clear_tools]}}
    if situation == "compact_now":
        return {"betas": ["compact-2026-09-04"], "compaction": {
            "type": "summarize",
            "instructions": "Summarize so the conversation can continue. Preserve every order ID, amount, date and promise exactly."}}
    if situation == "auto_compact":
        return {"betas": ["compact-2026-01-12"], "context_management": {"edits": [
            {"type": "compact_20260112", "trigger": {"type": "input_tokens", "value": 120_000}}]}}
    raise ValueError(f"unknown situation: {situation}")
```

</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
def window_used(usage) -> int:
    return (usage.input_tokens + (usage.cache_read_input_tokens or 0)
            + (usage.cache_creation_input_tokens or 0) + usage.output_tokens)


def choose_session(changed_files: list[str] | None, compare_approaches: bool = False, few: int = 3) -> str:
    if compare_approaches:
        return "fork"
    if changed_files is None or len(changed_files) > few:
        return "fresh_with_summary"
    return "resume_with_changes" if changed_files else "resume"
```

</details>

<details><summary>Part C — sample answer outline</summary>

**Support.** Request = cached `tools` + policy (breakpoint after the policy) → compacted summary → last ~6 turns verbatim → CRM results trimmed at ingest to task fields with a raw-record reference → `<case_facts>` with one entry per issue (IDs, amounts, dates, plan, customer expectations, promises; source + turn; unverified claims labeled; conflicts flagged) in the newest user turn (verified CRM facts may go in a turn-scoped system message; customer claims stay in the user turn) → customer message. Budget per layer, `count_tokens` before each call, all four `usage` fields after. At the budget: server-side compaction with value-preserving instructions; the facts block stays outside. In long tool loops: `clear_tool_uses_20250919` with `clear_at_least` and `exclude_tools=["get_customer"]`; no client-side edits to old turns (preserved thinking). Next day: a new session seeded with a structured summary, with facts re-verified against the CRM (or resume plus an explicit change list if only a known record changed).

**Pipeline.** Per-agent JSON state (status, key findings with source/date/method, coverage, gaps) + manifest, both written atomically after every status change. Restart: load manifest, skip completed, re-run in-progress idempotently, inject completed findings. Synthesis input: key findings first, `<document>` sections, task last.

**Metrics:** tokens/turn, cache-read share, compactions per chat, late-turn fact-accuracy evals, wrong-issue actions, pipeline resumes and minutes saved.

</details>

## Done

- Below 72%? Reread the README sections behind your ✗ marks, rerun the matching practice cells, and retake.
- Passed? Update the **Status** line at the top of the [README](README.md).

Next chapter: [Chapter 12 — Preserving Provenance](../12-provenance/README.md)